# 🦎 Lizard Species Prediction
**Thomas More – Deep Learning Project**

In dit notebook bouwen we een CNN om hagedissensoorten te classificeren.

### Stappen:
1. Imports & configuratie
2. EDA (Exploratory Data Analysis)
3. Data preprocessing & augmentation
4. Eigen CNN bouwen
5. Transfer learning (MobileNetV2)
6. Training visualiseren
7. Evaluatie (confusion matrix)
8. Kaggle submission genereren

## 1. 📦 Imports & Configuratie

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report

print(f"TensorFlow versie: {tf.__version__}")
print(f"GPU beschikbaar: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ── Paden aanpassen aan jouw projectstructuur ──
BASE_DIR   = Path("lizard-prediction-thomas-more")
TRAIN_DIR  = BASE_DIR / "train"
TEST_DIR   = BASE_DIR / "test"

# Hyperparameters
IMG_SIZE    = (128, 128)
BATCH_SIZE  = 32
EPOCHS_CNN  = 30
EPOCHS_TL   = 20
SEED        = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. 🔍 EDA – Exploratory Data Analysis

In [ ]:
# Klassen ophalen
classes = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
num_classes = len(classes)
class_to_idx = {c: i for i, c in enumerate(classes)}

print(f"Aantal klassen: {num_classes}")
print("Klassen:")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")

In [ ]:
# Aantal afbeeldingen per klasse
counts = {}
for cls in classes:
    imgs = list((TRAIN_DIR / cls).glob("*"))
    counts[cls] = len(imgs)

total = sum(counts.values())
print(f"Totaal aantal trainingsafbeeldingen: {total}\n")
for cls, cnt in counts.items():
    print(f"  {cls}: {cnt} afbeeldingen")

# Staafdiagram
plt.figure(figsize=(12, 5))
plt.bar(counts.keys(), counts.values(), color='steelblue', edgecolor='black')
plt.title("Aantal afbeeldingen per klasse")
plt.xlabel("Klasse")
plt.ylabel("Aantal")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Voorbeeldafbeeldingen per klasse
fig, axes = plt.subplots(2, num_classes, figsize=(num_classes * 3, 6))
for col, cls in enumerate(classes):
    imgs = list((TRAIN_DIR / cls).glob("*"))[:2]
    for row, img_path in enumerate(imgs):
        ax = axes[row][col] if num_classes > 1 else axes[row]
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.axis('off')
        if row == 0:
            ax.set_title(cls.replace('_', '\n'), fontsize=8)
plt.suptitle("Voorbeeldafbeeldingen per klasse", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Afbeeldingsgroottes bekijken
from PIL import Image

widths, heights = [], []
for cls in classes:
    for p in list((TRAIN_DIR / cls).glob("*"))[:20]:
        try:
            with Image.open(p) as im:
                widths.append(im.width)
                heights.append(im.height)
        except:
            pass

print(f"Breedte  – min: {min(widths)}, max: {max(widths)}, gemiddeld: {np.mean(widths):.0f}")
print(f"Hoogte   – min: {min(heights)}, max: {max(heights)}, gemiddeld: {np.mean(heights):.0f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(widths, bins=20, color='coral', edgecolor='black')
plt.title("Breedte verdeling")
plt.subplot(1, 2, 2)
plt.hist(heights, bins=20, color='skyblue', edgecolor='black')
plt.title("Hoogte verdeling")
plt.tight_layout()
plt.show()

## 3. 🔄 Data Preprocessing & Augmentation

In [ ]:
# Data augmentation – enkel tijdens training!
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False
)

print(f"Train samples: {train_generator.samples}")
print(f"Val samples:   {val_generator.samples}")
print(f"Klasse mapping: {train_generator.class_indices}")

In [ ]:
# Augmentatie visualiseren
sample_img_path = list((TRAIN_DIR / classes[0]).glob("*"))[0]
sample_img = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=IMG_SIZE)
sample_arr = tf.keras.preprocessing.image.img_to_array(sample_img)
sample_arr = sample_arr.reshape((1,) + sample_arr.shape)

aug_gen = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0][0].imshow(sample_img)
axes[0][0].set_title("Origineel")
axes[0][0].axis('off')

for i, batch in enumerate(aug_gen.flow(sample_arr, batch_size=1, seed=SEED)):
    ax = axes[(i + 1) // 5][(i + 1) % 5]
    ax.imshow(batch[0].astype('uint8'))
    ax.set_title(f"Aug {i+1}")
    ax.axis('off')
    if i >= 8:
        break

plt.suptitle("Data Augmentatie voorbeelden", fontsize=14)
plt.tight_layout()
plt.show()

## 4. 🧠 Eigen CNN bouwen

In [ ]:
def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        # Conv blok 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Conv blok 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Conv blok 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classificatie hoofd
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_cnn((*IMG_SIZE, 3), num_classes)
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cnn_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Start training eigen CNN...")
history_cnn = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=callbacks_cnn,
    verbose=1
)

## 5. 📈 Training visualiseren – Eigen CNN

In [ ]:
def plot_history(history, title="Training curves"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'],     label='Train accuracy', color='steelblue')
    ax1.plot(history.history['val_accuracy'], label='Val accuracy',   color='coral', linestyle='--')
    ax1.set_title(f'{title} – Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'],     label='Train loss', color='steelblue')
    ax2.plot(history.history['val_loss'], label='Val loss',   color='coral', linestyle='--')
    ax2.set_title(f'{title} – Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Eigen CNN")

## 6. 🔁 Transfer Learning

In [ ]:
# MobileNetV2 als base model (getraind op ImageNet)
base_model = keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Body bevriezen

# Eigen classificatie hoofd
inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

tl_model = keras.Model(inputs, outputs)
tl_model.summary()

In [ ]:
# Fase 1: enkel het hoofd trainen
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Nieuwe generators met MobileNetV2 preprocessing
tl_train_datagen = ImageDataGenerator(
    preprocessing_function=keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
    zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)
tl_val_datagen = ImageDataGenerator(
    preprocessing_function=keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

tl_train_gen = tl_train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=SEED
)
tl_val_gen = tl_val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=SEED, shuffle=False
)

callbacks_tl = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_tl_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Fase 1: hoofd trainen (body bevroren)...")
history_tl_1 = tl_model.fit(
    tl_train_gen, epochs=EPOCHS_TL,
    validation_data=tl_val_gen,
    callbacks=callbacks_tl, verbose=1
)

In [ ]:
# Fase 2: fine-tuning – laatste lagen van body ontgrendelen
base_model.trainable = True
fine_tune_at = 100  # Enkel lagen na layer 100 trainen
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # Lagere LR voor fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fase 2: fine-tuning...")
history_tl_2 = tl_model.fit(
    tl_train_gen, epochs=15,
    validation_data=tl_val_gen,
    callbacks=callbacks_tl, verbose=1
)

plot_history(history_tl_2, "Transfer Learning – Fine-tuning")

## 7. 📊 Evaluatie

In [ ]:
# Beste model laden en evalueren
best_model = keras.models.load_model('best_tl_model.keras')

val_loss, val_acc = best_model.evaluate(tl_val_gen, verbose=0)
print(f"\nValidatie accuracy (best TL model): {val_acc:.4f}")
print(f"Validatie loss:                     {val_loss:.4f}")

In [ ]:
# Voorspellingen op validatieset
tl_val_gen.reset()
y_pred_proba = best_model.predict(tl_val_gen, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = tl_val_gen.classes

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=classes))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=classes, yticklabels=classes
)
plt.title("Confusion Matrix – Transfer Learning Model", fontsize=14)
plt.xlabel("Voorspeld")
plt.ylabel("Werkelijk")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Genormaliseerde versie
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=classes, yticklabels=classes
)
plt.title("Confusion Matrix (genormaliseerd)", fontsize=14)
plt.xlabel("Voorspeld")
plt.ylabel("Werkelijk")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Vergelijking CNN vs Transfer Learning
cnn_val_loss, cnn_val_acc   = cnn_model.evaluate(val_generator, verbose=0)
tl_val_loss2, tl_val_acc2   = best_model.evaluate(tl_val_gen, verbose=0)

comparison = pd.DataFrame({
    'Model':    ['Eigen CNN', 'Transfer Learning'],
    'Val Acc':  [cnn_val_acc, tl_val_acc2],
    'Val Loss': [cnn_val_loss, tl_val_loss2]
})
print(comparison.to_string(index=False))

plt.figure(figsize=(8, 4))
bars = plt.bar(comparison['Model'], comparison['Val Acc'], color=['steelblue', 'coral'], edgecolor='black')
for bar, acc in zip(bars, comparison['Val Acc']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.title("Vergelijking Validatie Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 8. 🏆 Kaggle Submission

In [ ]:
# Label mapping: klasse-naam → integer label (1-gebaseerd zoals in sample submission)
# Pas dit aan als de Kaggle volgorde anders is!
idx_to_label = {i: i + 1 for i in range(num_classes)}  # 0→1, 1→2, ...

print("Label mapping (modelindex → Kaggle label):")
for model_idx, kaggle_label in idx_to_label.items():
    print(f"  {model_idx} ({classes[model_idx]}) → {kaggle_label}")

In [ ]:
# Testafbeeldingen verzamelen
test_images = sorted(TEST_DIR.glob("*"))
print(f"Aantal testafbeeldingen: {len(test_images)}")
print("Eerste paar:", [p.name for p in test_images[:5]])

In [ ]:
# Testgenerator (geen augmentatie, geen shuffle)
test_datagen = ImageDataGenerator(
    preprocessing_function=keras.applications.mobilenet_v2.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR.parent,          # parent van test
    classes=[TEST_DIR.name],  # enkel de test map
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

print(f"Test samples gevonden: {test_generator.samples}")

In [ ]:
# Voorspellingen genereren
test_generator.reset()
predictions = best_model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

# Bestandsnamen → id's ophalen
filenames = test_generator.filenames
# id = bestandsnaam zonder extensie (aanpassen als nummering anders is)
ids = [int(Path(f).stem) for f in filenames]

# Kaggle labels (1-based)
kaggle_labels = [idx_to_label[pc] for pc in predicted_classes]

submission = pd.DataFrame({'id': ids, 'label': kaggle_labels})
submission = submission.sort_values('id').reset_index(drop=True)

print(submission.head(10))
submission.to_csv('submission.csv', index=False)
print("\n✅ submission.csv opgeslagen!")

In [ ]:
# Verdeling van voorspellingen controleren
pred_counts = Counter(kaggle_labels)
plt.figure(figsize=(10, 4))
labels_sorted = sorted(pred_counts.keys())
plt.bar(
    [classes[l-1] for l in labels_sorted],
    [pred_counts[l] for l in labels_sorted],
    color='mediumseagreen', edgecolor='black'
)
plt.title("Verdeling voorspelde klassen (testset)")
plt.xlabel("Klasse")
plt.ylabel("Aantal")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\nSubmission statistieken:")
print(submission['label'].value_counts().sort_index())

## ✅ Samenvatting

| Stap | Beschrijving |
|------|--------------|
| EDA | Klassen, aantallen, voorbeeldafbeeldingen bekeken |
| Preprocessing | Rescaling, train/val split 80/20 |
| Augmentatie | Rotatie, flip, zoom, shift (enkel training) |
| Eigen CNN | 3 Conv-blokken + GlobalAveragePooling + Dense |
| Transfer Learning | MobileNetV2 (ImageNet) + fine-tuning |
| Evaluatie | Loss/accuracy curves, confusion matrix, classification report |
| Submission | submission.csv gegenereerd voor Kaggle |

### 💡 Tips voor hogere Kaggle score
- Grotere `IMG_SIZE` (bijv. 224×224) proberen
- `EfficientNetB3` of `ResNet50` als base model
- Learning rate schedule (CosineDecay)
- Class weights gebruiken bij ongebalanceerde data
- TTA (Test Time Augmentation) toepassen